# SecureRAG Tutorial 3: Strict gRPC API

This notebook uses the strict typed gRPC backend (`grpc://127.0.0.1:50051`) backed by the Rust-native gRPC server binary.

Start the server before running the cells:

```bash
source .venv/bin/activate
cargo build --manifest-path securerag-rs/Cargo.toml --bin securerag_grpc_server
./securerag-rs/target/debug/securerag_grpc_server --host 127.0.0.1 --port 50051
```


In [1]:
from securerag.backend_client import create_backend

In [2]:
backend = create_backend("grpc://127.0.0.1:50051")

docs = [
    {"doc_id": "q3", "text": "Q3 risk report highlights vendor concentration and delayed remediation.", "metadata": {}},
    {"doc_id": "policy", "text": "Security policy requires quarterly risk treatment tracking and owner assignment.", "metadata": {}},
]

chunks = backend.chunk(docs=docs, chunk_size=120, overlap=32)
len(chunks), chunks[0].keys()

(2, dict_keys(['metadata', 'text', 'doc_id']))

In [3]:
key = backend.sse_generate_key()
prepared = backend.sse_prepare_chunks(
    chunks=chunks,
    key=key,
    scheme="structured",
    use_bigrams=True,
)
index = backend.build_index(protocol="EncryptedSearch", chunks=prepared)
index

{'index_id': '6e79eb93-bda1-41a0-aced-fc7c79905742', 'doc_count': 2}

In [4]:
q_terms = backend.sse_encrypt_structured_terms(
    text="Summarize key risks in Q3",
    key=key,
    use_bigrams=True,
)
rows = backend.structured_search(index_id=index["index_id"], struct_terms=q_terms, top_k=3)
rows

[{'score': 0.08333333333333333,
  'metadata': {},
  'text': 'Q3 risk report highlights vendor concentration and delayed remediation.',
  'doc_id': 'q3'},
 {'score': 0.037037037037037035,
  'metadata': {},
  'text': 'Security policy requires quarterly risk treatment tracking and owner assignment.',
  'doc_id': 'policy'}]